# Inversion-only: bypass the feature-extraction step

If you already have a CSV of morphology features (e.g. from a
different segmenter or a different feature library), feed it
directly into `infer_from_features`. Required columns:

- `spheroid_id`
- `total_area`
- `equivalent_diameter`
- `solidity`
- `perimeter`
- `circularity`

Any other columns are passed through. One row per spheroid (or per
frame; the matcher does not distinguish - if you pass frame-resolved
rows the matcher treats each row as an independent observation and
the posterior summary will be over all rows for that spheroid_id).

In [1]:
import pandas as pd

from cll_cpm_inversion import (
    OPERATIONAL_FEATURES, PARAMS,
    infer_from_features,
    load_synthetic_library,
    load_identifiability,
    load_sobol_indices,
    feature_weights_from_sobol,
)

## Worked example: feed three library samples back in

A useful sanity check: a synthetic vector from the library should
match itself at rank 0 and yield a tight posterior around the truth.

In [2]:
lib = load_synthetic_library()
picks = lib.iloc[[10, 200, 400]].copy()
picks.index = ["lib_010", "lib_200", "lib_400"]
features_df = picks[OPERATIONAL_FEATURES].reset_index().rename(
    columns={"index": "spheroid_id"})
features_df

,spheroid_id,total_area,equivalent_diameter,solidity,perimeter,circularity
0,lib_010,175024.888889,472.068374,0.928203,3015.336037,0.243380
1,lib_200,159417.000000,450.528008,0.992046,1547.378105,0.836695
2,lib_400,134879.555556,414.407925,0.524342,53563.717668,0.000591


In [3]:
summary = infer_from_features(features_df, k=20)
summary.round(3)

,spheroid_id,parameter,median,q05,q95,q25,q75,n_matches,loo_r2,identifiability
0,lib_010,width,17.000,9.800,25.100,14.750,25.000,20,0.513,weakly identifiable
1,lib_010,temp,62.738,40.097,69.012,53.613,67.301,20,0.188,unidentifiable
2,lib_010,lambda,0.435,0.131,1.887,0.254,0.685,20,0.193,unidentifiable
3,lib_010,contact,31.242,21.480,43.052,28.562,39.855,20,0.319,weakly identifiable
4,lib_010,cm_adhesion,18.801,12.599,22.705,15.738,18.896,20,0.581,weakly identifiable
5,lib_010,contact_no,4.000,1.000,5.000,3.000,5.000,20,0.010,unidentifiable
6,lib_010,neighbor,3.500,2.000,6.050,2.750,5.000,20,-0.107,unidentifiable
7,lib_200,width,11.500,6.000,19.000,6.750,12.000,20,0.513,weakly identifiable
8,lib_200,temp,42.207,14.718,61.740,31.941,58.176,20,0.188,unidentifiable
9,lib_200,lambda,0.299,0.102,4.601,0.162,2.376,20,0.193,unidentifiable


Compare posterior median to ground truth:

In [4]:
truth = picks[PARAMS].reset_index().rename(columns={"index": "spheroid_id"})
truth_long = truth.melt(id_vars="spheroid_id",
                        var_name="parameter", value_name="truth")
compare = summary.merge(truth_long, on=["spheroid_id", "parameter"])
compare[["spheroid_id", "parameter", "truth", "median",
         "q05", "q95", "identifiability"]].round(2)

,spheroid_id,parameter,truth,median,q05,q95,identifiability
0,lib_010,width,17.00,17.00,9.80,25.10,weakly identifiable
1,lib_010,temp,46.20,62.74,40.10,69.01,unidentifiable
2,lib_010,lambda,0.25,0.43,0.13,1.89,unidentifiable
3,lib_010,contact,39.86,31.24,21.48,43.05,weakly identifiable
4,lib_010,cm_adhesion,18.80,18.80,12.60,22.71,weakly identifiable
5,lib_010,contact_no,4.00,4.00,1.00,5.00,unidentifiable
6,lib_010,neighbor,5.00,3.50,2.00,6.05,unidentifiable
7,lib_200,width,10.00,11.50,6.00,19.00,weakly identifiable
8,lib_200,temp,57.61,42.21,14.72,61.74,unidentifiable
9,lib_200,lambda,0.30,0.30,0.10,4.60,unidentifiable


Notice that even when feeding the library back into itself, the
unidentifiable parameters (`temp`, `lambda`, `contact_no`,
`neighbor`) are not recovered to their true values - the matcher
averages over the 20 most morphologically-similar synthetic samples,
and those samples have very different values of the unidentifiable
parameters. This is the practical signature of unidentifiability:
the feature-to-parameter mapping does not constrain those
parameters.

## Inspecting the matcher internals

The Sobol indices and per-feature weights that drive the matcher
are all loadable.

In [5]:
load_sobol_indices().head()

,feature,parameter,S1,ST,S1_conf,ST_conf
0,total_area,width,0.719678,0.927492,0.177615,0.156783
1,total_area,temp,0.017460,0.008154,0.018299,0.002449
2,total_area,contact,0.036073,0.066842,0.030240,0.023661
3,total_area,neighbor,-0.001418,0.005279,0.007250,0.001586
4,total_area,contact_no,0.023926,0.082798,0.035506,0.031190


In [6]:
weights = feature_weights_from_sobol()
pd.Series(weights, name="weight").to_frame().round(4)

,weight
total_area,0.2076
equivalent_diameter,0.2154
solidity,0.1995
perimeter,0.2019
circularity,0.1756


`circularity` carries the most weight because its mean total-order
Sobol index across the 7 parameters is the largest in this library.
This is what promotes $J_{cc}$ from unidentifiable (under uniform
weights, R^2 < 0.3) to weakly identifiable.